# Notebook 05: 認証の脆弱性

認証（Authentication）はシステムの入り口です。
ここに脆弱性があると、攻撃者はユーザーになりすまして全てにアクセスできます。

## 学習目標
1. パスワードの安全な保存方法（ハッシュ化 + ソルト）を理解する
2. セッション管理の仕組みと脆弱性を学ぶ
3. JWT の基本と注意点を知る

---

**この知識は防御側として活用してください**

## 1. パスワードの保存

### 危険: 平文保存
```
users テーブル:
| username | password     |
|----------|--------------|
| alice    | password123  |  ← そのまま保存！
| bob      | qwerty       |
```
データベースが漏洩すると全パスワードが一瞬で流出します。

### 少しまし: ハッシュ化
```
| username | password_hash                            |
|----------|------------------------------------------|
| alice    | ef92b778bafe...  (SHA-256 of password123) |
```
→ でもレインボーテーブル攻撃で元のパスワードが分かる場合がある

### 安全: ハッシュ + ソルト
```
| username | salt     | password_hash                   |
|----------|----------|---------------------------------|
| alice    | x9f3k2.. | hash(salt + password)           |
```
→ ソルト（ランダムな文字列）を加えることで同じパスワードでもハッシュ値が異なる

In [ ]:
import hashlib
import os

# === 危険: 平文保存 ===
print("=== 1. 平文保存（絶対にダメ） ===")
password = "password123"
print(f"保存される値: {password}")
print("→ DB漏洩 = 全パスワード流出")
print()

# === まし: 単純ハッシュ ===
print("=== 2. SHA-256 ハッシュのみ ===")
hash_value = hashlib.sha256(password.encode()).hexdigest()
print(f"password123 → {hash_value}")
# 同じパスワードは常に同じハッシュ
hash_value2 = hashlib.sha256("password123".encode()).hexdigest()
print(f"同じ？ {hash_value == hash_value2}")
print("→ レインボーテーブルで逆引き可能")
print()

# === 安全: ソルト + ハッシュ ===
print("=== 3. ソルト + ハッシュ ===")
salt1 = os.urandom(16).hex()
salt2 = os.urandom(16).hex()
hash_salted1 = hashlib.sha256((salt1 + password).encode()).hexdigest()
hash_salted2 = hashlib.sha256((salt2 + password).encode()).hexdigest()
print(f"alice: salt={salt1[:8]}..., hash={hash_salted1[:16]}...")
print(f"bob:   salt={salt2[:8]}..., hash={hash_salted2[:16]}...")
print(f"同じパスワードでもハッシュが異なる: {hash_salted1 != hash_salted2}")
print("→ レインボーテーブル攻撃が無効化される")

In [ ]:
import hashlib
import os
import time

# === 推奨: bcrypt / PBKDF2 ===
# 実際のアプリでは専用のパスワードハッシュ関数を使う

# PBKDF2 の例（Python標準ライブラリ）
print("=== PBKDF2（推奨） ===")
password = "password123"
salt = os.urandom(16)

start = time.time()
dk = hashlib.pbkdf2_hmac('sha256', password.encode(), salt, 100_000)
elapsed = time.time() - start

print(f"パスワード: {password}")
print(f"ソルト: {salt.hex()[:16]}...")
print(f"ハッシュ: {dk.hex()[:32]}...")
print(f"計算時間: {elapsed:.3f}秒")
print("→ 意図的に遅くすることでブルートフォース攻撃を困難にする")
print()

# 検証
print("=== パスワード検証 ===")
dk_check = hashlib.pbkdf2_hmac('sha256', "password123".encode(), salt, 100_000)
print(f"正しいパスワード: {dk == dk_check}")
dk_wrong = hashlib.pbkdf2_hmac('sha256', "wrong".encode(), salt, 100_000)
print(f"間違ったパスワード: {dk == dk_wrong}")

## 2. セッション管理

HTTP はステートレス（状態を持たない）なので、
ログイン状態を維持するためにセッション管理が必要です。

### 仕組み
```
1. ユーザーがログイン
2. サーバーがセッションIDを発行（ランダムな文字列）
3. セッションIDをCookieに保存
4. 以降のリクエストでCookieが自動送信される
5. サーバーがセッションIDでユーザーを識別
```

### セッション管理の脆弱性
- **セッション固定攻撃**: 攻撃者が決めたセッションIDを使わせる
- **セッションハイジャック**: XSS等でセッションIDを窃取
- **予測可能なセッションID**: 連番や日時ベース → 推測可能

In [ ]:
import secrets
import time

# === 脆弱なセッションID ===
print("=== 脆弱なセッションID ===")

# パターン1: 連番
print("連番: session_1, session_2, session_3")
print("→ 次のIDが予測できる")
print()

# パターン2: タイムスタンプベース
import hashlib
weak_session = hashlib.md5(str(time.time()).encode()).hexdigest()
print(f"タイムスタンプベース: {weak_session}")
print("→ 時刻が分かれば推測できる")
print()

# === 安全なセッションID ===
print("=== 安全なセッションID ===")
secure_session = secrets.token_hex(32)
print(f"暗号学的乱数: {secure_session}")
print(f"エントロピー: 256ビット（推測は事実上不可能）")
print()

# 比較
print("=== セッションIDの安全性 ===")
print(f"連番:         session_42        → 推測: 簡単")
print(f"MD5(time):    {weak_session[:20]}... → 推測: 可能")
print(f"secrets:      {secure_session[:20]}... → 推測: 不可能")

## 3. JWT（JSON Web Token）

JWT はセッション管理の代替手段としてよく使われます。
サーバー側でセッション情報を保持せず、トークン自体に情報を含めます。

### JWT の構造
```
ヘッダー.ペイロード.署名

eyJhbGciOiJIUzI1NiJ9.eyJ1c2VyIjoiYWxpY2UiLCJyb2xlIjoidXNlciJ9.XXXXX
```

各部分は Base64 エンコードされた JSON:
- **ヘッダー**: アルゴリズム指定（例: `{"alg": "HS256"}`）
- **ペイロード**: ユーザー情報（例: `{"user": "alice", "role": "user"}`）
- **署名**: ヘッダー + ペイロードの改ざん検出用

In [ ]:
import base64
import json
import hmac
import hashlib

def b64url_encode(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).rstrip(b'=').decode()

def b64url_decode(s: str) -> bytes:
    padding = 4 - len(s) % 4
    return base64.urlsafe_b64decode(s + '=' * padding)

# JWT を手動で作成
secret = "my_secret_key"

# ヘッダー
header = {"alg": "HS256", "typ": "JWT"}
header_b64 = b64url_encode(json.dumps(header).encode())

# ペイロード
payload = {"user": "alice", "role": "user"}
payload_b64 = b64url_encode(json.dumps(payload).encode())

# 署名
message = f"{header_b64}.{payload_b64}"
signature = hmac.new(secret.encode(), message.encode(), hashlib.sha256).digest()
signature_b64 = b64url_encode(signature)

token = f"{header_b64}.{payload_b64}.{signature_b64}"

print("=== JWT の構造 ===")
print(f"ヘッダー:  {header_b64}")
print(f"           → {json.loads(b64url_decode(header_b64))}")
print(f"ペイロード: {payload_b64}")
print(f"           → {json.loads(b64url_decode(payload_b64))}")
print(f"署名:      {signature_b64}")
print(f"\n完全なJWT:\n{token}")

In [ ]:
# JWT の脆弱性: alg=none 攻撃

print("=== JWT の脆弱性: alg=none ===")
print()

# 攻撃者がペイロードを改ざん
fake_header = {"alg": "none", "typ": "JWT"}
fake_payload = {"user": "admin", "role": "admin"}  # roleを昇格！

fake_header_b64 = b64url_encode(json.dumps(fake_header).encode())
fake_payload_b64 = b64url_encode(json.dumps(fake_payload).encode())
fake_token = f"{fake_header_b64}.{fake_payload_b64}."

print(f"正規トークン: {token[:50]}...")
print(f"偽造トークン: {fake_token}")
print()
print(f"偽造ヘッダー: {json.loads(b64url_decode(fake_header_b64))}")
print(f"偽造ペイロード: {json.loads(b64url_decode(fake_payload_b64))}")
print()
print("脆弱なサーバーは alg=none を受け入れてしまい、")
print("署名なしで admin 権限を取得される可能性がある")
print()

# 安全な検証
def verify_jwt_safe(token: str, secret: str) -> dict | None:
    """安全な JWT 検証"""
    parts = token.split('.')
    if len(parts) != 3:
        return None
    
    header = json.loads(b64url_decode(parts[0]))
    
    # alg=none を拒否！
    if header.get('alg') != 'HS256':
        print(f"  拒否: alg={header.get('alg')} は許可されていない")
        return None
    
    # 署名を検証
    message = f"{parts[0]}.{parts[1]}"
    expected_sig = hmac.new(secret.encode(), message.encode(), hashlib.sha256).digest()
    actual_sig = b64url_decode(parts[2])
    
    if not hmac.compare_digest(expected_sig, actual_sig):
        print("  拒否: 署名が一致しない")
        return None
    
    return json.loads(b64url_decode(parts[1]))

print("=== 安全な検証 ===")
print(f"正規トークン: {verify_jwt_safe(token, secret)}")
print(f"偽造トークン: {verify_jwt_safe(fake_token, secret)}")

## まとめ

| 項目 | 危険 | 安全 |
|------|------|------|
| パスワード保存 | 平文 / 単純ハッシュ | bcrypt / PBKDF2 + ソルト |
| セッションID | 連番 / 推測可能 | `secrets.token_hex()` |
| JWT | `alg=none` 許可 | アルゴリズム固定 + 署名検証 |
| Cookie | 通常 | `httponly`, `secure`, `samesite` |

## 演習

1. `hashlib.pbkdf2_hmac` を使ったパスワード登録・検証システムを作ってみよう
2. JWT のペイロードを Base64 デコードして中身を確認してみよう
3. `secrets.token_hex(32)` で生成したセッションIDのエントロピー（ビット数）を計算してみよう
4. labs/vulnerable_app の Lab 2 で SQLi によるログインbypass を試し、安全な認証との違いを考えてみよう